# Import necessary libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss
from sklearn.ensemble import RandomForestClassifier



import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display



ModuleNotFoundError: No module named 'xgboost'

# Phase 1: Project Setup & Target Analysis

# Read file with Labels

In [4]:
df_train = pd.read_csv('../data/conversion_data_train.csv')
df_test = pd.read_csv('../data/conversion_data_test.csv')
print('Set with labels (our train+test) :', df_train.shape)
print('Set with labels (our test) :', df_test.shape)

Set with labels (our train+test) : (284580, 6)
Set with labels (our test) : (31620, 5)


# Explore DATASET

In [6]:
df_train.head(10)

,country,age,new_user,source,total_pages_visited,converted
0,China,22,1,Direct,2,0
1,UK,21,1,Ads,3,0
2,Germany,20,0,Seo,14,1
3,US,23,1,Seo,3,0
4,US,28,1,Direct,3,0
5,US,29,0,Seo,7,0
6,US,30,1,Direct,4,0
7,UK,38,1,Ads,2,0
8,UK,26,1,Seo,4,0
9,UK,31,0,Seo,5,0


In [7]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284580 entries, 0 to 284579
Data columns (total 6 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   country              284580 non-null  object
 1   age                  284580 non-null  int64 
 2   new_user             284580 non-null  int64 
 3   source               284580 non-null  object
 4   total_pages_visited  284580 non-null  int64 
 5   converted            284580 non-null  int64 
dtypes: int64(4), object(2)
memory usage: 13.0+ MB


In [8]:
df_train.describe(include='all')


,country,age,new_user,source,total_pages_visited,converted
count,284580,284580.000000,284580.000000,284580,284580.000000,284580.000000
unique,4,NaN,NaN,3,NaN,NaN
top,US,NaN,NaN,Seo,NaN,NaN
freq,160124,NaN,NaN,139477,NaN,NaN
mean,NaN,30.564203,0.685452,NaN,4.873252,0.032258
std,NaN,8.266789,0.464336,NaN,3.341995,0.176685
min,NaN,17.000000,0.000000,NaN,1.000000,0.000000
25%,NaN,24.000000,0.000000,NaN,2.000000,0.000000
50%,NaN,30.000000,1.000000,NaN,4.000000,0.000000
75%,NaN,36.000000,1.000000,NaN,7.000000,0.000000


In [9]:
df_train.shape

(284580, 6)

In [10]:
df_train.converted.value_counts()

converted
0    275400
1      9180
Name: count, dtype: int64

In [11]:
#le dataset est extrêmement déséquilibré: 97% de non conversion contre 3% de conversion
df_train['converted'].value_counts(normalize=True)



converted
0    0.967742
1    0.032258
Name: proportion, dtype: float64

* **Class Imbalance:** Only ~3.2% of users converted. This heavy imbalance confirms that `Accuracy` is misleading, making `F1-Score` our primary evaluation metric.
* **Next Step:** Preprocessing pipeline (One-Hot Encoding, Feature Scaling) and baseline Logistic Regression training.

# Phase 2: Preprocessing & Logistic Regression Baseline

- Evaluation Metric & Business Logic
We analyze the model performance using the full Confusion Matrix components:
* **True Negatives (TN):** Correctly identified non-converters.
* **True Positives (TP):** Correctly identified converters (our target).
* **False Positives (FP):** Users predicted to convert but did not. (Wastes marketing budget).
* **False Negatives (FN):** Users predicted to drop out but converted. (Missed revenue opportunities).

Due to severe **Class Imbalance** (~3.2% converters discovered in Train EDA), standard `Accuracy` is heavily biased. Thus, we optimize for the **F1-Score** to perfectly balance Precision (minimizing FP) and Recall (minimizing FN).

- Preprocessing Strategy
To feed our linear Logistic Regression, we apply a clear pipeline:
1. **One-Hot Encoding:** For categorical inputs (`country`, `source`, `new_user`).
2. **Feature Scaling (StandardScaler):** For continuous variables (`age`, `total_pages_visited`) to normalize magnitudes.

In [13]:
#delete converted column from dataset and assign it to y
X = df_train.drop('converted', axis=1)
y = df_train['converted']

In [14]:
# Train/Validation Split (80% / 20%)
#stratify to maintain the same proportion of classes in train and val sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [15]:
#preprocessing

numeric_features = ['age', 'total_pages_visited']
categorical_features = ['country', 'source', 'new_user']

#Pipeline Sklearn
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(drop='first') # drop='first' évite la colinéarité

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

In [16]:
#Baseline Logistic Regression model as a reference
baseline_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

#Training model
baseline_model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [17]:
#Evaluation on the validation set
y_val_pred = baseline_model.predict(X_val)

print("METRICS & CONFUSION MATRIX (Validation Set):")
print("-" * 50)
print(f"F1-Score: {f1_score(y_val, y_val_pred):.4f}")
print("\nConfusion Matrix Layout:")
print("[[ TN   FP ]")
print(" [ FN   TP ]]")
print("\nComputed Confusion Matrix:")
print(confusion_matrix(y_val, y_val_pred))

print("\nDetailed Classification Report:")
print(classification_report(y_val, y_val_pred))

METRICS & CONFUSION MATRIX (Validation Set):
--------------------------------------------------
F1-Score: 0.7678

Confusion Matrix Layout:
[[ TN   FP ]
 [ FN   TP ]]

Computed Confusion Matrix:
[[54886   194]
 [  571  1265]]

Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99     55080
           1       0.87      0.69      0.77      1836

    accuracy                           0.99     56916
   macro avg       0.93      0.84      0.88     56916
weighted avg       0.99      0.99      0.99     56916



**Confusion Matrix Deep Dive:**
* **True Negatives (TN) = 54,886** & **True Positives (TP) = 1,265**: The model successfully identifies the vast majority of non-converters and captures a solid portion of real conversions.
* **False Positives (FP) = 194 (Precision = 87%)**: Very low error rate here. When the model predicts a conversion, it is highly reliable. Marketing campaign costs are well protected.
* **False Negatives (FN) = 571 (Recall = 69%)**: This is our main bottleneck. The model completely misses 571 users who actually converted. This represents a substantial loss in potential revenue.

**Metric Evaluation:**
* **F1-Score = 0.7678**: While the global accuracy is 99% due to the heavy class imbalance (~3.2%), the F1-Score gives us the true benchmark of our model's balance on the minority class. 
* **Conclusion:** The baseline is strong but conservative. We must introduce regularization to find the optimal feature weights and potentially reduce the number of False Negatives (FN).

## Phase3 : Moving to Regularized Models: Ridge & Lasso

To optimize our linear baseline and prevent any overfitting, we implement **L1 (Lasso)** and **L2 (Ridge)** regularization. 
* **Lasso** will help us perform automatic feature selection by driving useless weights to exactly zero.
* **Ridge** will shrink coefficients uniformly to handle potential multicollinearity between features.

## Introducing Lasso (L1) Regularization

In [18]:
# Lasso L1 Regularization
lasso_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(penalty='l1', solver='liblinear', C=1.0, random_state=42))
])

print("🏋️ Training Logistic Regression with Lasso (L1)...")
lasso_model.fit(X_train, y_train)

🏋️ Training Logistic Regression with Lasso (L1)...


,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [19]:
# Predictions and Evaluation Metrics
y_val_pred_l1 = lasso_model.predict(X_val)

print("\n📊 LASSO (L1) METRICS & CONFUSION MATRIX:")
print("-" * 50)
print(f"Lasso F1-Score: {f1_score(y_val, y_val_pred_l1):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_l1))
print("\nClassification Report:")
print(classification_report(y_val, y_val_pred_l1))


📊 LASSO (L1) METRICS & CONFUSION MATRIX:
--------------------------------------------------
Lasso F1-Score: 0.7683

Confusion Matrix:
[[54883   197]
 [  568  1268]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99     55080
           1       0.87      0.69      0.77      1836

    accuracy                           0.99     56916
   macro avg       0.93      0.84      0.88     56916
weighted avg       0.99      0.99      0.99     56916



In [20]:
# Features Importance Extraction (Coefficients)
cat_encoder = preprocessor.named_transformers_['cat']
encoded_cat_features = list(cat_encoder.get_feature_names_out(categorical_features))
all_features = numeric_features + encoded_cat_features
coefficients = lasso_model.named_steps['classifier'].coef_[0]

importance_df = pd.DataFrame({
    'Feature': all_features,
    'Coefficient': coefficients
}).sort_values(by='Coefficient', key=abs, ascending=False)

print("\n🎯 LASSO FEATURE IMPORTANCE:")
print("-" * 50)
print(importance_df)


🎯 LASSO FEATURE IMPORTANCE:
--------------------------------------------------
               Feature  Coefficient
2      country_Germany     3.685332
3           country_UK     3.505575
4           country_US     3.162052
1  total_pages_visited     2.526409
7           new_user_1    -1.690578
0                  age    -0.598512
5        source_Direct    -0.206023
6           source_Seo    -0.035825


** Metric Comparison:**
* **F1-Score (0.7683):** The Lasso regularization yields a minor improvement compared to our unregularized baseline (`0.7678`).
* **Confusion Matrix:** With **197 False Positives** and **568 False Negatives**, the model behavior remains stable. It heavily prioritizes protecting the budget (low FP) but still misses too many converters (high FN).

**Feature Importance Insights:**
* **Geographics & Behavior:** Living in Germany, the UK, or the US has the strongest positive impact on conversion compared to the baseline country (China). High `total_pages_visited` is also a powerful driver of conversion.
* **Negative Drivers:** Being a new user (`new_user_1` coefficient: `-1.69`) and increasing `age` (`-0.59`) significantly decrease the probability of conversion.
* **Feature Selection:** No coefficients were driven to exact zero, meaning all features hold some predictive signal for the linear boundary.

## Introducing Ridge (L2) Regularization

To complement our L1 analysis, we now train a **Ridge (L2)** Logistic Regression. While Lasso attempts to eliminate features, Ridge shrinks coefficients uniformly, which helps handle any multi-collinearity without discarding information.

In [21]:
# Definition and training of the Ridge (L2) Pipeline
ridge_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(penalty='l2', C=1.0, random_state=42))
])

print("🏋️ Training Logistic Regression with Ridge (L2)...")
ridge_model.fit(X_train, y_train)

🏋️ Training Logistic Regression with Ridge (L2)...


,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [22]:
# Predictions and Evaluation Metrics
y_val_pred_l2 = ridge_model.predict(X_val)

print("\n📊 RIDGE (L2) METRICS & CONFUSION MATRIX:")
print("-" * 50)
print(f"Ridge F1-Score: {f1_score(y_val, y_val_pred_l2):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_l2))
print("\nClassification Report:")
print(classification_report(y_val, y_val_pred_l2))


📊 RIDGE (L2) METRICS & CONFUSION MATRIX:
--------------------------------------------------
Ridge F1-Score: 0.7678

Confusion Matrix:
[[54886   194]
 [  571  1265]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99     55080
           1       0.87      0.69      0.77      1836

    accuracy                           0.99     56916
   macro avg       0.93      0.84      0.88     56916
weighted avg       0.99      0.99      0.99     56916



**Metric & Confusion Matrix Analysis:**

* **F1-Score (0.7678):** The Ridge regularization performs exactly like our initial unregularized baseline model.
* **Error Distribution:** We observe **194 False Positives (FP)** and **571 False Negatives (FN)**.
* **Conclusion on Linear Models:** Both Lasso and Ridge have reached a mathematical ceiling. The linear boundary of Logistic Regression cannot capture any more conversion signals, leaving our Recall stuck at 69%. To capture more True Positives and reduce the 571 False Negatives, we must move away from linear models and explore non-linear structures.

## Phase 4: Random Forest


To break the performance ceiling of linear boundaries, we transition to a **Random Forest Classifier**. 
Unlike Logistic Regression, Random Forest does not look for a straight line; it builds multiple decision trees based on non-linear thresholds (e.g., specific age groups or combinations of page visits and countries) to catch complex patterns and reduce False Negatives (FN).

In [25]:
# Definition and training of the Random Forest Pipeline
rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1))
])

print("Training Random Forest Classifier...")
rf_model.fit(X_train, y_train)

#  Predictions and Evaluation Metrics
y_val_pred_rf = rf_model.predict(X_val)

print("\n RANDOM FOREST METRICS & CONFUSION MATRIX:")
print("-" * 50)
print(f"Random Forest F1-Score: {f1_score(y_val, y_val_pred_rf):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_rf))
print("\nClassification Report:")
print(classification_report(y_val, y_val_pred_rf))

Training Random Forest Classifier...

 RANDOM FOREST METRICS & CONFUSION MATRIX:
--------------------------------------------------
Random Forest F1-Score: 0.7579

Confusion Matrix:
[[54882   198]
 [  595  1241]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99     55080
           1       0.86      0.68      0.76      1836

    accuracy                           0.99     56916
   macro avg       0.93      0.84      0.88     56916
weighted avg       0.99      0.99      0.99     56916



** Metric & Confusion Matrix Analysis:**
* **F1-Score (0.7579):** The Random Forest performance decreases slightly compared to the linear models (Lasso: `0.7683`, Ridge: `0.7678`).
* **Error Analysis:** We observe **198 False Positives (FP)** and **595 False Negatives (FN)**. The number of missed converters (FN) increased from 568 (Lasso) to 595.
* **Conclusion:** Tree-based bagging models like Random Forest tend to struggle with heavily imbalanced tabular datasets (~3.2% conversions) without explicit class weighting. The decision boundaries created by default trees are overfitting the majority class (`converted=0`). To counter this, we must transition to a powerful boosting architecture designed to focus iteratively on hard-to-predict errors: **XGBoost**.

## Phase 5: Final Optimization: Threshold Tuning on Random Forest

By default, the Random Forest uses a strict `0.5` probability threshold to classify a user as converted. Because our dataset is highly imbalanced (~3.2%), the model's raw predicted probabilities are naturally compressed toward low values. 

To reduce the high number of False Negatives (595 FN) and convert them into True Positives, we will perform a mathematical search to find the optimal decision threshold that maximizes our target metric, the **F1-Score**.

In [30]:
# probabilities of conversion on the validation set
y_val_proba_rf = rf_model.predict_proba(X_val)[:, 1]

best_rf_threshold = 0.5
best_rf_f1 = 0

# iterative search for the best threshold between 0.1 and 0.9
for thresh in np.arange(0.1, 0.9, 0.01):
    y_pred_temp = (y_val_proba_rf >= thresh).astype(int)
    current_f1 = f1_score(y_val, y_pred_temp)
    
    if current_f1 > best_rf_f1:
        best_rf_f1 = current_f1
        best_rf_threshold = thresh

print(" OPTIMAL THRESHOLD SEARCH FOR RANDOM FOREST:")
print("-" * 50)
print(f"Best Found Threshold : {best_rf_threshold:.2f}")
print(f"Maximized F1-Score   : {best_rf_f1:.4f}")

# confusion matrix and classification report with the new threshold 
y_val_pred_rf_opt = (y_val_proba_rf >= best_rf_threshold).astype(int)

print("\n OPTIMIZED RANDOM FOREST METRICS:")
print("-" * 50)
print("Confusion Matrix with New Threshold:")
print(confusion_matrix(y_val, y_val_pred_rf_opt))
print("\nClassification Report with New Threshold:")
print(classification_report(y_val, y_val_pred_rf_opt))

 OPTIMAL THRESHOLD SEARCH FOR RANDOM FOREST:
--------------------------------------------------
Best Found Threshold : 0.41
Maximized F1-Score   : 0.7667

 OPTIMIZED RANDOM FOREST METRICS:
--------------------------------------------------
Confusion Matrix with New Threshold:
[[54812   268]
 [  528  1308]]

Classification Report with New Threshold:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99     55080
           1       0.83      0.71      0.77      1836

    accuracy                           0.99     56916
   macro avg       0.91      0.85      0.88     56916
weighted avg       0.99      0.99      0.99     56916



**Threshold Shift & Impact:**
* **Optimal Threshold (0.41):** Lowering the decision cutoff from `0.50` to `0.41` allowed the model to be more sensitive to the minority class (`converted=1`).
* **Confusion Matrix Evolution:**
  * **False Negatives (FN) dropped from 595 to 528**: By lowering the barrier, we successfully captured **67 additional real converters** (True Positives went up from 1241 to 1308), recovering lost revenue.
  * **False Positives (FP) increased from 198 to 268**: As expected, being more aggressive leads to a small increase in targeting errors (Precision dropped from 86% to 83%).
* **Metric Evaluation:** The optimized F1-Score rises to **0.7667**. This proves that shifting the classification threshold is a robust technique to battle severe class imbalance without changing the underlying mathematical structure of the trees.

# Phase 6: Final Project Synthesis & Business Takeaways

## 1. Modeling Strategy Review
* We established a solid linear **Logistic Regression baseline** (`F1 = 0.7678`), which proved highly resistant to overfitting.
* **Lasso (L1)** and **Ridge (L2)** regularizations confirmed that the linear boundary was mathematically optimized at around `0.7683`.
* Non-linear **Random Forest** initially struggled with the 3.2% class imbalance, but **Threshold Tuning at 0.41** successfully unlocked a higher **Recall (71%)**, capturing more converters and significantly reducing False Negatives.

## 2. Strategic Business Recommendations
Based on our Feature Importance and model observations, we suggest the following actions to improve the conversion rate:
1. **Target Demographics:** Younger users have a significantly higher conversion probability. Marketing budget should be reallocated to target younger cohorts.
2. **Geographic Focus:** Users located in Germany, the UK, and the US exhibit massive positive coefficients compared to China. Localization and specific ad campaigns should be reinforced in these three western markets.
3. **User Retention:** New users show a heavy negative impact on conversion (`-1.69`). Implementing an onboarding email sequence or a first-purchase discount could bridge the gap and convert new visitors into returning customers.
4. **Behavioral Lever:** `total_pages_visited` is the strongest predictor. Web UX/UI teams should optimize the interface to keep users engaged and browsing more pages (e.g., related products widgets, gamification).

##############################
# FINAL MODEL FOR SUBMISSION
##############################


In [31]:
#conversion probabilities for the test set using the trained Random Forest model
test_probabilities = rf_model.predict_proba(df_test)[:, 1]

#application of the optimal threshold (0.41) to get the final classes 0 or 1
test_predictions = (test_probabilities >= 0.41).astype(int)

#creation of the submission DataFrame as required by the statement
submission_df = pd.DataFrame({
    'converted': test_predictions
})

#Exportation of conversion predictions to a CSV file (without row indices)
submission_path = "../data/conversion_predictions_submission.csv"
submission_df.to_csv(submission_path, index=False)

print(f"🎉 Success! Final predictions file exported to: {submission_path}")
print("\n👀 Preview of your submission file (first 5 predictions):")
print(submission_df.head())


🎉 Success! Final predictions file exported to: ../data/conversion_predictions_submission.csv

👀 Preview of your submission file (first 5 predictions):
   converted
0          1
1          0
2          0
3          0
4          0
